# EIBP Slice Builder: GraphML Parsing
# PTP Testing 

This works in tandem with the Graph Analyzer code previously written, if desired (not required). That code can be found here: https://github.com/pjw7904/Graph-Analyzer/tree/develop

Any graphml file comprised of basic node and edge tags will be able to work with this parser. Each node will be set up as a FABRIC node and any edge will be set up as a l2network between the specified nodes. Currently, this code does not consider extended LANs with more than two nodes.

This was written to take advantage of existing graphml files, as opposed to the FABRIC-enhanced graphml RSPEC files that contain hardware properties that go beyond the basic topological information.

In [ ]:
from ipaddress import ip_address, IPv4Address, IPv4Network
import ipaddress
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager
try:
    fablib = fablib_manager()
                     
    fablib.show_config()
except Exception as e:
    print(f"Exception: {e}")

In [ ]:
slice_name=f"Slice for KNIT7 Precision Timing Tutorial"
sites = []
avoid_sites = []
sites = fablib.get_random_sites(count=4,filter_function=lambda x:x['ptp_capable'] is True, avoid=(avoid_sites))

print (f"PTP Capable sites selected are {sites}")

## Input Required Information

| Variable | Use |
| --- | --- |
| SLICE_NAME    | Name of slice you want to create. Please make sure a slice with that name does not already exist. |
| SITE_NAME     | Name of the FABRIC site you want the nodes to be reserved at. This code does not consider inter-site situations, the entire topology is reserved on a single slice. |
| GRAPH_PATH    | Path to the graphml file you want to use to create a topology. |
| HAS_CLIENTS   | Enter True if clients are present in topology, if not, False. These nodes and the networks connecting them utilizes alternative naming and addressing structures. |
| CLIENT_PREFIX | The naming prefix given to each node (currently, this is required if the topology does have clients) |
| MEAS_ADD      | Enter True if measurements are to be taken on the slice. This requires the inclusion of a separate measurement node |

In [1]:
SLICE_NAME = "EIBP_PTP_Large_13"
SITE_NAME = "TACC"
## CHANGE DIRECTORY PATHS 
GRAPH_PATH = "/home/fabric/work/NShenoy/Failure_HW_SW/EIBP_OSPF/EIBP_hw_sw/graphs/13Node.graphml"
HAS_CLIENTS = True
CLIENT_PREFIX = "ipnode"
MEAS_ADD = False

## Import the FABlib Library and Confirm the Configuration is Correct

In [2]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

try:
    fablib = fablib_manager()

    fablib.show_config()
except Exception as e:
    print(f"Exception: {e}")

User: nxsvks@rit.edu bastion key is valid!
Configuration is valid


Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
Token File,/home/fabric/.tokens.json
Project ID,fec0d0d8-a7a8-4eac-b091-87f7914af796
Bastion Host,bastion.fabric-testbed.net
Bastion Username,nxsvks_0000031803
Bastion Private Key File,/home/fabric/work/fabric_config/Nirmala
Slice Public Key File,/home/fabric/work/fabric_config/SliceKey.pub
Slice Private Key File,/home/fabric/work/fabric_config/SliceKey


## Parse the GraphML for the Topology and Create the Slice

The minidom library is used to parse the graphml. It assumes proper use of the node and edge tags. Examples of valid tags can be seen below. The name of the node is the node's id. Furthermore, it does not matter which node is the source and destination, it is parsed as an undirected graph and there will be issues if you try and create another network between the same two nodes.

    <node id="L1" />
    <node id="S1" />
    <edge source="L1" target="S1" />


In [3]:
import xml.dom.minidom
from collections import Counter
try:
    #Create the slice
    slice = fablib.new_slice(name=SLICE_NAME)

    # Create dictionary to store nodes
    nodeDict = {}

    # Create dictionary to store information to dump into a file
    logFile = {"name": SLICE_NAME, "site": SITE_NAME, "hasClients": HAS_CLIENTS, "meas": MEAS_ADD}

    # Use XML parser to parse the GraphML file
    docs = xml.dom.minidom.parse(GRAPH_PATH)

    # Find all nodes via the node tag, add each to the slice with Rocky Linux as its base
    nodes = docs.getElementsByTagName("node")
    for node in nodes:
        # Grab the node name and determine if it is a client node based on its name prefix
        nodeName = node.getAttribute("id")
        isClient = True if HAS_CLIENTS and nodeName.startswith(CLIENT_PREFIX) else False # Check for compute/client nodes

        ### LOG FILE INFO
        logFile[nodeName] = {"isClient": isClient, "ssh": None, "networks": {}}

        # Add node to the slice
        nodeInfo = slice.add_node(name=nodeName, cores=4, ram=8, image='default_rocky_8', site=SITE_NAME)

        # nodeDict = 0 -> FABRIC node object, 1 -> is a client/server (True) or not (False)
        nodeDict[nodeName] = {"nodeInfo": nodeInfo, "isClient": isClient}

        print(f'Added node {nodeName}')

    # Find all edges via the edge tag, add each to the slice via an L2Bridge connecting the node interfaces
    edges = docs.getElementsByTagName("edge")
    for edge in edges:
        # grab nodes x and y in edge (x,y)
        source = edge.getAttribute("source")
        target = edge.getAttribute("target")

        # Create an interface name for each interface in the network
        sourceIntfName = f"intf-{target}"
        targetIntfName = f"intf-{source}"

        # Name a network based on if it is a user-facing LAN (edge) or P2P links in the core of the network (core)
        networkPrefix = "edge" if nodeDict[source]["isClient"]  or nodeDict[target]["isClient"] else "core"
        networkName = f'{networkPrefix}-{source}-{target}'

        # Add a NIC for each node that is a part of the edge
        sourceIntf = nodeDict[source]["nodeInfo"].add_component(model='NIC_Basic', name=sourceIntfName).get_interfaces()[0]
        targetIntf = nodeDict[target]["nodeInfo"].add_component(model='NIC_Basic', name=targetIntfName).get_interfaces()[0]

        # Add a L2 network between the interfaces
        slice.add_l2network(name=networkName, interfaces=[sourceIntf, targetIntf], type="L2Bridge")

        ### LOG FILE INFO
        logFile[source]["networks"][networkName] = {"neighbor": target}
        logFile[target]["networks"][networkName] = {"neighbor": source}

        print(f'Added edge {source}-{target}')

except Exception as e:
    print(f"Exception: {e}")

Added node C1
Added node C2
Added node C3
Added node D1
Added node D2
Added node D3
Added node D4
Added node D5
Added node A1
Added node A2
Added node A3
Added node A4
Added node A5
Added node ipnode-1
Added node ipnode-2
Added node ipnode-3
Added node ipnode-4
Added node ipnode-5
Added edge C1-C2
Added edge C1-C3
Added edge C1-D1
Added edge C1-D2
Added edge C1-D3
Added edge C2-C3
Added edge C2-D3
Added edge C2-D4
Added edge C2-D5
Added edge C3-D1
Added edge C3-D2
Added edge C3-D4
Added edge C3-D5
Added edge D1-A1
Added edge D1-A2
Added edge D2-A1
Added edge D2-A3
Added edge D3-A2
Added edge D3-A4
Added edge D4-A3
Added edge D4-A5
Added edge D5-A4
Added edge D5-A5
Added edge A1-ipnode-1
Added edge A2-ipnode-2
Added edge A3-ipnode-3
Added edge A4-ipnode-4
Added edge A5-ipnode-5


## Submit the Slice

In [4]:
%%time
import json

try:
    # Submit Slice Request
    print(f'Submitting the new slice, "{SLICE_NAME}"...')
    slice.submit()
    print(f'{SLICE_NAME} creation done.')

except Exception as e:
    print(f"Slice Fail: {e}")
    traceback.print_exc()


Retry: 21, Time: 910 sec


ID,3fc716b8-daf5-40fe-b884-7365844c83fa
Name,EIBP_PTP_Large_13
Lease Expiration (UTC),2026-04-21 14:25:26 +0000
Lease Start (UTC),2026-04-20 14:25:26 +0000
Project ID,fec0d0d8-a7a8-4eac-b091-87f7914af796
State,StableOK
Email,nxsvks@rit.edu
UserId,03baf9ee-7e80-4dc3-8012-d62bd4db65fb


ID,Name,Cores,RAM,Disk,Image,Image Type,Host,Site,Username,Management IP,State,Error,SSH Command,Public SSH Key File,Private SSH Key File
d4daf041-70bf-47bf-ad91-dd2ff98e5dc0,A1,4,8,10,default_rocky_8,qcow2,tacc-w3.fabric-testbed.net,TACC,rocky,2605:2800:2011:201:f816:3eff:fe16:9adb,Active,,ssh -i /home/fabric/work/fabric_config/SliceKey -F /home/fabric/work/fabric_config/ssh_config rocky@2605:2800:2011:201:f816:3eff:fe16:9adb,/home/fabric/work/fabric_config/SliceKey.pub,/home/fabric/work/fabric_config/SliceKey
23d101cb-7561-4a2e-bccc-b865e868d82d,A2,4,8,10,default_rocky_8,qcow2,tacc-w3.fabric-testbed.net,TACC,rocky,2605:2800:2011:201:f816:3eff:fead:7226,Active,,ssh -i /home/fabric/work/fabric_config/SliceKey -F /home/fabric/work/fabric_config/ssh_config rocky@2605:2800:2011:201:f816:3eff:fead:7226,/home/fabric/work/fabric_config/SliceKey.pub,/home/fabric/work/fabric_config/SliceKey
13888b6d-2460-4714-9b36-a76106d37868,A3,4,8,10,default_rocky_8,qcow2,tacc-w2.fabric-testbed.net,TACC,rocky,2605:2800:2011:201:f816:3eff:fe47:4c68,Active,,ssh -i /home/fabric/work/fabric_config/SliceKey -F /home/fabric/work/fabric_config/ssh_config rocky@2605:2800:2011:201:f816:3eff:fe47:4c68,/home/fabric/work/fabric_config/SliceKey.pub,/home/fabric/work/fabric_config/SliceKey
44b5f17b-0750-40e0-88d6-230648a0b88e,A4,4,8,10,default_rocky_8,qcow2,tacc-w2.fabric-testbed.net,TACC,rocky,2605:2800:2011:201:f816:3eff:fea0:a86a,Active,,ssh -i /home/fabric/work/fabric_config/SliceKey -F /home/fabric/work/fabric_config/ssh_config rocky@2605:2800:2011:201:f816:3eff:fea0:a86a,/home/fabric/work/fabric_config/SliceKey.pub,/home/fabric/work/fabric_config/SliceKey
6a5c5a73-27e2-413f-bc70-934fa61bdce8,A5,4,8,10,default_rocky_8,qcow2,tacc-w2.fabric-testbed.net,TACC,rocky,2605:2800:2011:201:f816:3eff:feaf:1c89,Active,,ssh -i /home/fabric/work/fabric_config/SliceKey -F /home/fabric/work/fabric_config/ssh_config rocky@2605:2800:2011:201:f816:3eff:feaf:1c89,/home/fabric/work/fabric_config/SliceKey.pub,/home/fabric/work/fabric_config/SliceKey
e9d295e9-7e30-4244-bddc-c6ce01d8ce14,C1,4,8,10,default_rocky_8,qcow2,tacc-w4.fabric-testbed.net,TACC,rocky,2605:2800:2011:201:f816:3eff:feb8:4ceb,Active,,ssh -i /home/fabric/work/fabric_config/SliceKey -F /home/fabric/work/fabric_config/ssh_config rocky@2605:2800:2011:201:f816:3eff:feb8:4ceb,/home/fabric/work/fabric_config/SliceKey.pub,/home/fabric/work/fabric_config/SliceKey
b5c925ce-0134-4302-9bdd-e6d94a2789cc,C2,4,8,10,default_rocky_8,qcow2,tacc-w3.fabric-testbed.net,TACC,rocky,2605:2800:2011:201:f816:3eff:feb0:75f6,Active,,ssh -i /home/fabric/work/fabric_config/SliceKey -F /home/fabric/work/fabric_config/ssh_config rocky@2605:2800:2011:201:f816:3eff:feb0:75f6,/home/fabric/work/fabric_config/SliceKey.pub,/home/fabric/work/fabric_config/SliceKey
cf317147-b6bc-4552-ad35-71454c732540,C3,4,8,10,default_rocky_8,qcow2,tacc-w3.fabric-testbed.net,TACC,rocky,2605:2800:2011:201:f816:3eff:fec9:556e,Active,,ssh -i /home/fabric/work/fabric_config/SliceKey -F /home/fabric/work/fabric_config/ssh_config rocky@2605:2800:2011:201:f816:3eff:fec9:556e,/home/fabric/work/fabric_config/SliceKey.pub,/home/fabric/work/fabric_config/SliceKey
bbea065e-98ad-4c80-a43b-54f9a802b22a,D1,4,8,10,default_rocky_8,qcow2,tacc-w3.fabric-testbed.net,TACC,rocky,2605:2800:2011:201:f816:3eff:fe48:47e2,Active,,ssh -i /home/fabric/work/fabric_config/SliceKey -F /home/fabric/work/fabric_config/ssh_config rocky@2605:2800:2011:201:f816:3eff:fe48:47e2,/home/fabric/work/fabric_config/SliceKey.pub,/home/fabric/work/fabric_config/SliceKey
c8b710c4-3a67-4b81-af15-6a8a714d166d,D2,4,8,10,default_rocky_8,qcow2,tacc-w3.fabric-testbed.net,TACC,rocky,2605:2800:2011:201:f816:3eff:fe2b:1949,Active,,ssh -i /home/fabric/work/fabric_config/SliceKey -F /home/fabric/work/fabric_config/ssh_config rocky@2605:2800:2011:201:f816:3eff:fe2b:1949,/home/fabric/work/fabric_config/SliceKey.pub,/home/fabric/work/fabric_config/SliceKey


ID,Name,Layer,Type,Site,Subnet,Gateway,State,Error
7785894b-b253-4a2b-b9a2-13ba75a4affe,core-C1-C2,L2,L2Bridge,TACC,None,None,Active,
a1d82602-283f-4af4-88c5-c56c82feb0c2,core-C1-C3,L2,L2Bridge,TACC,None,None,Active,
ff0f82a7-4377-4185-9a33-2421c2753c29,core-C1-D1,L2,L2Bridge,TACC,None,None,Active,
760a0fcc-4ced-4e81-9dd6-28b02da4f294,core-C1-D2,L2,L2Bridge,TACC,None,None,Active,
aef5bce2-f2fd-4a84-88a0-1ba31b5c1acf,core-C1-D3,L2,L2Bridge,TACC,None,None,Active,
d4e77076-c941-4ac4-809e-166816ee6cdb,core-C2-C3,L2,L2Bridge,TACC,None,None,Active,
86a974ed-a175-4cb9-8173-e1cc05e322b7,core-C2-D3,L2,L2Bridge,TACC,None,None,Active,
17926bf4-660a-46f0-8fe7-4f19c9e32361,core-C2-D4,L2,L2Bridge,TACC,None,None,Active,
9964d3cf-57fd-4bfe-8a6e-5a56ef82787e,core-C2-D5,L2,L2Bridge,TACC,None,None,Active,
b50e94f5-f808-479d-baa3-97f4d52b7cc8,core-C3-D1,L2,L2Bridge,TACC,None,None,Active,


Name,Short Name,Node,Network,Bandwidth,Mode,VLAN,MAC,Physical Device,Device,IP Address,Numa Node,Switch Port
C1-intf-C2-p1,p1,C1,core-C1-C2,100,config,,26:23:E3:8E:96:CB,eth3,eth3,None,4,HundredGigE0/0/0/11
C1-intf-D2-p1,p1,C1,core-C1-D2,100,config,,26:42:54:3F:69:48,eth4,eth4,None,4,HundredGigE0/0/0/11
C1-intf-D1-p1,p1,C1,core-C1-D1,100,config,,26:1F:D3:25:EE:26,eth2,eth2,None,4,HundredGigE0/0/0/11
C1-intf-C3-p1,p1,C1,core-C1-C3,100,config,,26:0C:FA:6D:5F:8E,eth1,eth1,None,4,HundredGigE0/0/0/11
C1-intf-D3-p1,p1,C1,core-C1-D3,100,config,,26:9C:F4:56:28:34,eth5,eth5,None,4,HundredGigE0/0/0/11
C2-intf-D3-p1,p1,C2,core-C2-D3,100,config,,3E:B5:3D:E2:6E:B5,eth1,eth1,None,4,HundredGigE0/0/0/9
C2-intf-C1-p1,p1,C2,core-C1-C2,100,config,,42:65:39:40:25:90,eth3,eth3,None,4,HundredGigE0/0/0/9
C2-intf-C3-p1,p1,C2,core-C2-C3,100,config,,42:6A:01:F5:02:17,eth4,eth4,None,4,HundredGigE0/0/0/9
C2-intf-D5-p1,p1,C2,core-C2-D5,100,config,,42:00:43:CC:A7:08,eth2,eth2,None,4,HundredGigE0/0/0/9
C2-intf-D4-p1,p1,C2,core-C2-D4,100,config,,46:71:90:E9:B0:A4,eth5,eth5,None,4,HundredGigE0/0/0/9



Time to print interfaces 1082 seconds
EIBP_PTP_Large_13 creation done.
CPU times: user 7min 54s, sys: 3.57 s, total: 7min 57s
Wall time: 18min 12s


In [6]:
nodes = slice.get_nodes()
for node in nodes:
    print (f"{node.get_name()} is hosted on {node.get_host()}")
    ad = fablib.get_site_advertisement(node.get_site())
    print (f"PTP Capable: { ad.flags.ptp}\n")

C1 is hosted on tacc-w4.fabric-testbed.net
PTP Capable: False

C2 is hosted on tacc-w3.fabric-testbed.net
PTP Capable: False

C3 is hosted on tacc-w3.fabric-testbed.net
PTP Capable: False

D1 is hosted on tacc-w3.fabric-testbed.net
PTP Capable: False

D2 is hosted on tacc-w3.fabric-testbed.net
PTP Capable: False

D3 is hosted on tacc-w3.fabric-testbed.net
PTP Capable: False

D4 is hosted on tacc-w3.fabric-testbed.net
PTP Capable: False

D5 is hosted on tacc-w3.fabric-testbed.net
PTP Capable: False

A1 is hosted on tacc-w3.fabric-testbed.net
PTP Capable: False

A2 is hosted on tacc-w3.fabric-testbed.net
PTP Capable: False

A3 is hosted on tacc-w2.fabric-testbed.net
PTP Capable: False

A4 is hosted on tacc-w2.fabric-testbed.net
PTP Capable: False

A5 is hosted on tacc-w2.fabric-testbed.net
PTP Capable: False

ipnode-1 is hosted on tacc-w2.fabric-testbed.net
PTP Capable: False

ipnode-2 is hosted on tacc-w2.fabric-testbed.net
PTP Capable: False

ipnode-3 is hosted on tacc-w2.fabric-testbe

## Add Basic IPv4 Addressing (optional)

We run this for testing purpose.

In this system, 192.168.0.0/16 is the address space for all interfaces on the FABRIC slice.

Each network is a /24 subnet of this network. Edge networks have client/compute devices with lower address (ex: .1) and networking nodes with higher addresses.

In [7]:
from ipaddress import IPv4Network

def updateMeasNetworkName(node, nodeName, intfName):
    if "meas" not in nodeName:
        node.execute(command=f"sudo ip link set dev {intfName} down")
        node.execute(command=f"sudo ip link set dev {intfName} name meas")
        node.execute(command=f"sudo ip link set dev meas up")

        print(f"\t{nodeName} {intfName} renamed meas")
    else:
        print(f"\tMeasurement node not modified")

    return

# Start with a 1 in the third octet
third_octet = 1

# For each network in the slice
for network in slice.get_networks():
    # Determine if the current network is an edge network
    network_name = network.get_name()
    is_edge_network = network_name.startswith("edge")

    # Print configuration information for edge networks
    if is_edge_network:
        network_address = f'192.168.{third_octet}.0/24'
        current_ip_network = IPv4Network(network_address)
        host_ip_list = list(current_ip_network.hosts())  # Exclude network and broadcast addresses

        print(f"Configuring network {network_name} with IPv4 Network {network_address}")
        third_octet += 1
    # For measurement networks, print configuration without network address
    elif "meas" in network_name:
        print(f"Configuring network {network_name}")

    # For each interface in the network
    for intf in network.get_interfaces():
        intf_name = intf.get_physical_os_interface_name()
        node = intf.get_node()
        node_name = node.get_name()

        # Check if the node is a client node and not a measurement node
        if is_edge_network and "meas" not in network_name:
            # If this is an edge network, assign IP address only to client nodes
            if nodeDict[node_name]["isClient"]:
                current_ipv4_address = host_ip_list.pop(0)
            else:
                current_ipv4_address = host_ip_list.pop()

            # Add the address to the node
            intf.ip_addr_add(addr=current_ipv4_address, subnet=current_ip_network)
            print(f"\t{node_name} {intf.get_device_name()} = {current_ipv4_address}")

            # Update log file information
            logFile[node_name]["networks"][network_name]["subnet"] = str(current_ip_network)
            logFile[node_name]["networks"][network_name]["ipv4"] = str(current_ipv4_address)

        # If it's a measurement network, update interface names
        elif "meas" in network_name:
            updateMeasNetworkName(node, node_name, intf_name)

Configuring network edge-A1-ipnode-1 with IPv4 Network 192.168.1.0/24
	A1 eth3 = 192.168.1.254
	ipnode-1 eth1 = 192.168.1.1
Configuring network edge-A2-ipnode-2 with IPv4 Network 192.168.2.0/24
	A2 eth1 = 192.168.2.254
	ipnode-2 eth1 = 192.168.2.1
Configuring network edge-A3-ipnode-3 with IPv4 Network 192.168.3.0/24
	A3 eth3 = 192.168.3.254
	ipnode-3 eth1 = 192.168.3.1
Configuring network edge-A4-ipnode-4 with IPv4 Network 192.168.4.0/24
	ipnode-4 eth1 = 192.168.4.1
	A4 eth2 = 192.168.4.254
Configuring network edge-A5-ipnode-5 with IPv4 Network 192.168.5.0/24
	A5 eth1 = 192.168.5.254
	ipnode-5 eth1 = 192.168.5.1


## Log Topology Information

In [8]:
for node in slice.get_nodes():
    nodeName = node.get_name()

    if("meas" not in nodeName):
        logFile[nodeName]["ssh"] = node.get_ssh_command()

## LOG FILE INFO
with open(f"{SLICE_NAME}_slice_log.json", "w") as outfile:
    json.dump(logFile, outfile)

## Install and setup linuxptp package on nodes
Download the Ansible role to configure and install the LinuxPTP software. For more details regarding the steps performed in the playbook, please refer to the repo at https://github.com/fabric-testbed/ptp

In [9]:
pre_requisites = None

# Set Deployment tool repository details
repo_branch = 'main'
repo_name = 'ptp'
destination_folder = f"""/tmp/{repo_name}-{repo_branch}"""
clone_instructions = f"""
cd /tmp/;rm -rf /tmp/{repo_name}-{repo_branch};git clone --branch {repo_branch} https://github.com/fabric-testbed/{repo_name}.git {destination_folder};
"""

### Setting PTP Install Restrictions

* If you do not want all interfaces synchronized to PTP, add the name of interfaces to avoid as shown
* Management interfaces are not considered and are avoided by default
* If you do not want the system clock synchronized to PTP set the 'SYNC_SYSTEM_CLOCK' to False
* If you do not have any restrictions for a node, you can omit that node from the list

Example:
```
NODE_RESTRICTIONS = { 
   'node1' : { 'AVOID_IFACES': ['enp6s0'],'SYNC_SYSTEM_CLOCK': False},
   'node2' : { 'AVOID_IFACES': ['enp6s0','enp7s0']},
}
```

In [10]:
NODE_RESTRICTIONS = {}

### Restrict Ansible operation based on tags

* Possible values are ptp_stop,ptp_start,ptp_install 
* Only one tag is allowed
* If empty then all three are performed in the right sequence
* If NODE_RESTRICTIONS are applied along with the tags, the operations will not be performed on the AVOIDED INTERFACES

Example
```
ansible_tags = 'ptp_stop'
```

In [11]:
ansible_tags = ''

### Run Ansible playbook on each node

In [12]:
# Instruction to run ansible command from the node
ansible_instructions = f"""
cd {destination_folder}/ansible;ansible-playbook --connection=local --inventory 127.0.0.1, --limit 127.0.0.1 playbook_fabric_experiment_ptp.yml"""

#Create execute threads
execute_threads = {}

for node in nodes:
    if [ele for ele in ["rocky", "centos"] if (ele in node.get_image())]:
        pre_requisites = f"""
        sudo dnf -y install epel-release ; sudo dnf -y install ansible git;
        """
    elif [ele for ele in ["ubuntu", "debian"] if (ele in node.get_image())]:
        pre_requisites = f"""sudo apt-get update;sudo apt-get -y install ansible git;"""
    else:
        pre_requisites = None
    node_name = node.get_name()
    
    # Create JSON files for extra params that will be provided to ansible
    if node_name in NODE_RESTRICTIONS.keys():    
        extra_ansible_params = f""" --extra-vars @parameters.json""";
        with open('/tmp/'+node_name+'-parameters.json', 'w') as f:
            json.dump(NODE_RESTRICTIONS[node_name], f)
        print (f"Uploading install restrictions for {node_name}")    
        node.upload_file('/tmp/'+node_name+'-parameters.json',destination_folder+'/ansible/parameters.json')
    else:
        extra_ansible_params = ''
    if ansible_tags != '':
        extra_ansible_params = extra_ansible_params + ' --tags '+ansible_tags
        
    print (f"Running the PTP Deployment Ansible Playbook on {node.get_name()}")
    execute_threads[node] = node.execute_thread(\
                f"{pre_requisites}"\
                f"{clone_instructions}"\
                f"{ansible_instructions}"\
                f"{extra_ansible_params}",\
                output_file=f"/tmp/{node.get_name()}_ptpinstall.log"\
                )

    #Wait for results from threads
for node,thread in execute_threads.items():
    print(f"Waiting for result from node {node.get_name()}")
    stdout,stderr = thread.result()

print (f"Ansible Playbook run on all nodes completed\n")

Running the PTP Deployment Ansible Playbook on C1
Running the PTP Deployment Ansible Playbook on C2
Running the PTP Deployment Ansible Playbook on C3
Running the PTP Deployment Ansible Playbook on D1
Running the PTP Deployment Ansible Playbook on D2
Running the PTP Deployment Ansible Playbook on D3
Running the PTP Deployment Ansible Playbook on D4
Running the PTP Deployment Ansible Playbook on D5
Running the PTP Deployment Ansible Playbook on A1
Running the PTP Deployment Ansible Playbook on A2
Running the PTP Deployment Ansible Playbook on A3
Running the PTP Deployment Ansible Playbook on A4
Running the PTP Deployment Ansible Playbook on A5
Running the PTP Deployment Ansible Playbook on ipnode-1
Running the PTP Deployment Ansible Playbook on ipnode-2
Running the PTP Deployment Ansible Playbook on ipnode-3
Running the PTP Deployment Ansible Playbook on ipnode-4
Running the PTP Deployment Ansible Playbook on ipnode-5
Waiting for result from node C1
Waiting for result from node C2
Waitin

In [13]:
slice = fablib.get_slice(name = SLICE_NAME)
nodes = slice.get_nodes()
for node in nodes:
    nodeName = node.get_name()
    node_interfaces = node.get_interfaces()
    for interface_obj in node_interfaces:
        interface = interface_obj.get_device_name()
        print (f"Working on {nodeName}->{interface}")
        print (f" Starting PTP Synchronization on node->{interface}") 
        stdout,stderr = node.execute(f'sudo systemctl start phc2sys@{interface}.service;sleep 5')
        print (f" Get Time from node->{interface} CLOCK/PHC")
        stdout,stderr = node.execute("sudo ethtool -T "+interface+"|grep 'PTP Hardware Clock:'|awk '{print $4}'",quiet=True)
        ptp_index = stdout.strip()
        stdout,stderr = node.execute(f"sudo phc_ctl /dev/ptp{ptp_index} get;sudo phc_ctl /dev/ptp{ptp_index} cmp")
    print (f"Time Sync Operation Completed\n\n")

Working on C1->eth1
 Starting PTP Synchronization on node->eth1
 Get Time from node->eth1 CLOCK/PHC
phc_ctl[2789.241]: clock time is 1776698099.300850782 or Mon Apr 20 15:14:59 2026

phc_ctl[2789.254]: offset from CLOCK_REALTIME is 474ns

Working on C1->eth5
 Starting PTP Synchronization on node->eth5
 Get Time from node->eth5 CLOCK/PHC
phc_ctl[2797.560]: clock time is 1776698107.619984075 or Mon Apr 20 15:15:07 2026

phc_ctl[2797.572]: offset from CLOCK_REALTIME is 461ns

Working on C1->eth3
 Starting PTP Synchronization on node->eth3
 Get Time from node->eth3 CLOCK/PHC
phc_ctl[2805.364]: clock time is 1776698115.424405603 or Mon Apr 20 15:15:15 2026

phc_ctl[2805.377]: offset from CLOCK_REALTIME is 292ns

Working on C1->eth4
 Starting PTP Synchronization on node->eth4
 Get Time from node->eth4 CLOCK/PHC
phc_ctl[2813.524]: clock time is 1776698123.583960276 or Mon Apr 20 15:15:23 2026

phc_ctl[2813.537]: offset from CLOCK_REALTIME is 909ns

Working on C1->eth2
 Starting PTP Synchroniz